[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-14-capstone-finetune-deploy.ipynb#scrollTo=aa110001)

---
# Day 14 · Capstone — Fine-Tune a Transformer on a Custom NLP Task and Deploy to Spaces
**certified-journeys / huggingface-nlp-certified** · Day 14 · Capstone

> **Goal for today:** Fine-tune a BERT-family model on a chosen NLP task, evaluate it with the appropriate metric, push the model and tokenizer to the Hugging Face Hub with a complete model card, and deploy a live Gradio Space that accepts user text input and displays predictions with confidence scores.

## Capstone Overview

This notebook is structured as a complete, end-to-end project. Work through all seven stages in order:

| Stage | What you do |
|---|---|
| 1. Install & imports | One-time environment setup |
| 2. Task & dataset | Choose a task; load a Hub dataset (or custom CSV) |
| 3. Tokenize | Tokenize with `distilbert-base-uncased` or `bert-base-uncased` |
| 4. Fine-tune | Train 3 epochs with `Trainer`, `fp16=True`, `load_best_model_at_end=True` |
| 5. Evaluate | Report the correct metric (F1, accuracy, ROUGE) on the test split |
| 6. Push to Hub | `trainer.push_to_hub()` + write a complete model card |
| 7. Deploy Gradio | Build and launch a Gradio Space that loads your Hub model |

**Reference:** [HF NLP Course — Chapter 7 Walkthrough](https://huggingface.co/learn/nlp-course/chapter7/1)

> **Before you start:** Log in to the Hub. In Colab, run: `from huggingface_hub import notebook_login; notebook_login()`

In [ ]:
%pip install -q transformers datasets evaluate accelerate torch gradio huggingface_hub

## Stage 1 · Configure Your Capstone

Set your Hub username and choose a task. The defaults run a sentiment-classification fine-tune on SST-2, which trains in under 30 minutes on a free T4. You can swap `TASK`, `DATASET`, and `CHECKPOINT` for any compatible combination.

Supported combinations:

| TASK | DATASET | CHECKPOINT | METRIC |
|---|---|---|---|
| `text-classification` | `glue/sst2` | `distilbert-base-uncased` | accuracy |
| `text-classification` | `imdb` | `bert-base-uncased` | accuracy |
| `token-classification` | `conll2003` | `bert-base-uncased` | F1 |
| `summarization` | `cnn_dailymail` | `facebook/bart-base` | ROUGE-L |

The remainder of the notebook is parameterised on the constants below — change them once and re-run.

In [ ]:
# ── Capstone configuration — edit these values ───────────────────────────────
HF_USERNAME  = "your-username"          # your Hugging Face hub username
MODEL_SLUG   = "distilbert-sst2-capstone" # name for your Hub repo
CHECKPOINT   = "distilbert-base-uncased"  # base model to fine-tune
DATASET_NAME = "glue"                     # Hub dataset name
DATASET_CFG  = "sst2"                     # dataset config / subset
TEXT_COL     = "sentence"                 # column containing the input text
LABEL_COL    = "label"                    # column containing the label
NUM_LABELS   = 2                          # number of output classes
MAX_LEN      = 128                        # max tokenization length
TASK         = "text-classification"      # pipeline task string
METRIC_NAME  = "accuracy"                 # evaluate metric name
OUTPUT_DIR   = f"./{MODEL_SLUG}-output"   # local training output directory
HUB_REPO_ID  = f"{HF_USERNAME}/{MODEL_SLUG}"

print("Capstone configuration:")
print(f"  Base checkpoint : {CHECKPOINT}")
print(f"  Dataset         : {DATASET_NAME}/{DATASET_CFG}")
print(f"  Text column     : {TEXT_COL}")
print(f"  Label column    : {LABEL_COL}")
print(f"  Num labels      : {NUM_LABELS}")
print(f"  Max length      : {MAX_LEN}")
print(f"  Hub repo        : {HUB_REPO_ID}")

### What just happened?
- All downstream cells read from these constants — change them once, re-run the notebook, and you have a different fine-tuning project.
- **`HUB_REPO_ID`** is the identifier that `push_to_hub()` will use. Make sure `HF_USERNAME` matches your actual Hub account.
- `MAX_LEN=128` is deliberately conservative; SST-2 sentences rarely exceed 50 tokens, so truncation cost is zero here.
- For custom CSV data, replace the `load_dataset` call in Stage 2 with `load_dataset('csv', data_files={'train': 'train.csv', 'validation': 'val.csv'})`.

## Stage 2 · Load and Inspect the Dataset

We load from the Hub. For a custom CSV use:
```python
from datasets import load_dataset
ds = load_dataset('csv', data_files={'train': 'train.csv', 'test': 'test.csv'})
```

Always inspect the dataset before tokenizing:
- Check column names match `TEXT_COL` and `LABEL_COL`
- Verify the label distribution is not severely imbalanced
- Confirm a test split exists (or carve one from the training set)

| Key | SST-2 value | Your dataset |
|---|---|---|
| Train size | 67 349 | ? |
| Validation size | 872 | ? |
| Test size | 1 821 | ? |
| Labels | 0=negative, 1=positive | ? |

In [ ]:
from datasets import load_dataset
import collections

raw = load_dataset(DATASET_NAME, DATASET_CFG)
print("Dataset splits:")
for split, ds in raw.items():
    print(f"  {split:<12}: {len(ds):>6} rows | columns: {ds.column_names}")

# Inspect a sample
print("\nSample (train[0]):")
sample = raw["train"][0]
print(f"  {TEXT_COL}  : {sample[TEXT_COL]!r}")
print(f"  {LABEL_COL} : {sample[LABEL_COL]}")

# Label distribution (train)
train_labels = raw["train"][LABEL_COL]
dist = collections.Counter(train_labels)
total = len(train_labels)
print("\nLabel distribution (train):")
for label, count in sorted(dist.items()):
    print(f"  label {label}: {count:>6} ({count/total*100:.1f}%)")

### What just happened?
- `load_dataset` returns a `DatasetDict` with named splits — always verify all expected splits are present.
- **SST-2 is 56% positive / 44% negative** — balanced enough that accuracy is a fair metric; for highly imbalanced data prefer macro-F1.
- The column inspection step catches the most common data prep mistake: misnamed columns that cause a `KeyError` inside `map()`.
- If your dataset has no test split, use `raw['train'].train_test_split(test_size=0.1)` to create one.

## Stage 3 · Tokenize

Tokenization converts raw text into `input_ids`, `attention_mask`, and (for BERT) `token_type_ids`. Key decisions:

- **`max_length`**: controls sequence length budget — directly impacts GPU memory
- **`truncation=True`**: required to avoid errors on long inputs
- **`padding='max_length'`**: pads all sequences to `MAX_LEN` — faster but wastes compute; use `DataCollatorWithPadding` instead for dynamic padding
- **`batched=True`**: maps over batches of 1 000 rows at a time — much faster than row-by-row

```
Raw text  →  tokenizer()  →  input_ids + attention_mask  →  model
```

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize_fn(batch):
    """Map function: tokenize the text column with truncation."""
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
    )

# Apply tokenization to all splits
tokenized = raw.map(
    tokenize_fn,
    batched=True,
    remove_columns=[TEXT_COL],  # keep only token columns + label
)

# Dynamic padding per batch — more efficient than padding to MAX_LEN globally
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized dataset:")
for split, ds in tokenized.items():
    print(f"  {split:<12}: {len(ds):>6} rows | columns: {ds.column_names}")

# Inspect tokenized sample
ex = tokenized["train"][0]
print(f"\ninput_ids length  : {len(ex['input_ids'])}")
print(f"attention_mask sum: {sum(ex['attention_mask'])}  (non-padding tokens)")
print(f"label             : {ex['label']}")

### What just happened?
- `batched=True` processes 1 000 rows per call — typically 10–20× faster than row-by-row mapping.
- `remove_columns=[TEXT_COL]` drops the original text so the dataset only contains tensor-compatible columns.
- **`DataCollatorWithPadding` pads each batch to its longest sequence** — shorter sequences cost less compute than padding everything to `MAX_LEN`.
- The `attention_mask` shows how many real (non-padding) tokens the sample contains — the model ignores positions where `attention_mask == 0`.

## Stage 4 · Fine-Tune with Trainer

The `Trainer` API handles the training loop, evaluation, checkpointing, and optional Hub push. Key `TrainingArguments` for the capstone:

| Argument | Value | Why |
|---|---|---|
| `num_train_epochs` | 3 | Enough for convergence on most classification tasks |
| `fp16` | True | Halves memory; 1.5–2× faster on Nvidia GPUs |
| `load_best_model_at_end` | True | Returns the checkpoint with best eval metric |
| `evaluation_strategy` | "epoch" | Eval after each epoch to track learning curve |
| `save_strategy` | "epoch" | Must match `evaluation_strategy` for best model loading |
| `metric_for_best_model` | "accuracy" | What "best" means — change to "f1" for NER |
| `push_to_hub` | True | Auto-push to Hub after training |

> **Note on `fp16`:** If you are on a CPU-only runtime, set `fp16=False`. The Trainer will raise a warning and fall back automatically, but it is cleaner to set it explicitly.

In [ ]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"  # fp16 only works on GPU
print(f"Training on: {device} | fp16: {use_fp16}")

# Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=NUM_LABELS,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,      # increase to 4 if GPU OOM
    fp16=use_fp16,
    learning_rate=2e-5,                 # standard starting point for BERT fine-tuning
    weight_decay=0.01,                  # L2 regularisation
    warmup_ratio=0.1,                   # 10% of steps used for linear warmup
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_NAME,
    greater_is_better=True,
    push_to_hub=False,                  # set to True after login
    hub_model_id=HUB_REPO_ID,
    report_to="none",
    logging_steps=100,
)

print(f"Output dir      : {OUTPUT_DIR}")
print(f"Effective batch : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
import evaluate
import numpy as np

# Load the metric — swap METRIC_NAME for your task
metric = evaluate.load(METRIC_NAME)

def compute_metrics(eval_pred):
    """Called by Trainer after each evaluation step."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # argmax over class dimension
    return metric.compute(predictions=predictions, references=labels)

# Use a small training subset in demo mode to keep runtime short.
# Remove the .select() calls to train on the full dataset.
DEMO_MODE = True
if DEMO_MODE:
    train_ds = tokenized["train"].select(range(2000))
    eval_ds  = tokenized["validation"].select(range(500))
    print("Demo mode: using 2 000 train / 500 val samples")
else:
    train_ds = tokenized["train"]
    eval_ds  = tokenized["validation"]
    print(f"Full mode: {len(train_ds)} train / {len(eval_ds)} val samples")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready. Starting training…")
trainer.train()

### What just happened?
- `AutoModelForSequenceClassification` adds a linear classification head on top of the BERT encoder and randomly initialises it — the rest of the weights are pre-trained.
- **`load_best_model_at_end=True` requires `evaluation_strategy == save_strategy`** — both set to `"epoch"` here.
- `warmup_ratio=0.1` linearly ramps the learning rate from 0 → `learning_rate` over the first 10% of training steps, preventing large gradient updates on random initial weights.
- `compute_metrics` receives `(logits, labels)` from the Trainer; `np.argmax` converts logit vectors to predicted class indices.
- `DEMO_MODE=True` uses 2 000 samples to complete in ~2 minutes — set to `False` for the real capstone submission.

## Stage 5 · Evaluate on the Test Split

Always evaluate on a **held-out test split**, not the validation split used during training. The validation split influenced model selection (via `load_best_model_at_end`) — evaluating on it gives an optimistic estimate.

| Task type | Correct metric | When to prefer it |
|---|---|---|
| Binary classification | Accuracy | Balanced classes |
| Multi-class | Macro-F1 | Imbalanced classes |
| NER / token classification | Entity-level F1 (seqeval) | Standard for CoNLL |
| Summarization | ROUGE-L | Overlap-based proxy for quality |
| Generation | BLEU / BERTScore | Translation / generation quality |

In [ ]:
import numpy as np

# SST-2 test split labels are -1 (unlabeled), so we evaluate on validation
# For your own dataset, swap validation for test if a labeled test split exists
test_split = "validation"
test_ds = tokenized[test_split]
if DEMO_MODE:
    test_ds = test_ds.select(range(500))

print(f"Evaluating on {test_split} split ({len(test_ds)} samples)…")
results = trainer.evaluate(eval_dataset=test_ds)

print("\nFinal evaluation results:")
for k, v in results.items():
    print(f"  {k:<35}: {v:.4f}" if isinstance(v, float) else f"  {k:<35}: {v}")

# Extract the primary metric for the model card
primary_metric_key = f"eval_{METRIC_NAME}"
primary_score = results.get(primary_metric_key, None)
if primary_score is not None:
    print(f"\nFinal {METRIC_NAME}: {primary_score:.4f}")

### What just happened?
- `trainer.evaluate()` runs the model in eval mode (no gradient computation) and calls `compute_metrics` on the predictions.
- The results dict includes `eval_loss`, `eval_accuracy` (or your metric), `eval_runtime`, and `eval_samples_per_second`.
- **Record `primary_score` — this goes into the model card** in the next stage.
- SST-2 fine-tuned DistilBERT typically achieves 90–92% accuracy on the full validation set; 3-epoch demo mode on 2 000 samples will be lower (~85–88%).

## Stage 6 · Save Locally, Write a Model Card, and Push to Hub

A complete model card (README.md on the Hub) must include:

1. **Task and dataset** — what the model does and what it was trained on
2. **Base checkpoint** — which pretrained model was fine-tuned
3. **Training decisions** — learning rate, batch size, epochs, fp16
4. **Final eval metric** — the number from Stage 5
5. **Usage snippet** — runnable code showing how to use the model
6. **Limitations** — known failure modes or out-of-distribution warnings

> **Tip:** Document every training decision in your model card: why you chose the base model, what learning rate and batch size you used, and what the final eval metric was — this is what makes a Hub model genuinely useful to others.

In [ ]:
import os
from pathlib import Path

# ── Step 6a: Save model and tokenizer locally ────────────────────────────────
trainer.save_model(OUTPUT_DIR)          # saves weights + config
tokenizer.save_pretrained(OUTPUT_DIR)   # must match the model
print(f"Saved model + tokenizer to: {OUTPUT_DIR}")
print("Files:", sorted(os.listdir(OUTPUT_DIR))[:10])  # first 10 files

# ── Step 6b: Write a model card ─────────────────────────────────────────────
score_str = f"{primary_score:.4f}" if primary_score is not None else "see trainer_state.json"

model_card = f"""---
language: en
license: apache-2.0
tags:
  - text-classification
  - bert
  - huggingface-nlp-certified
datasets:
  - {DATASET_NAME}
metrics:
  - {METRIC_NAME}
model-index:
  - name: {MODEL_SLUG}
    results:
      - task:
          type: {TASK}
        dataset:
          name: {DATASET_NAME}/{DATASET_CFG}
          type: {DATASET_NAME}
        metrics:
          - type: {METRIC_NAME}
            value: {score_str}
---

# {MODEL_SLUG}

Fine-tuned [{CHECKPOINT}](https://huggingface.co/{CHECKPOINT}) on
[{DATASET_NAME}/{DATASET_CFG}](https://huggingface.co/datasets/{DATASET_NAME}) for
**{TASK}**.

## Training details

| Parameter | Value |
|---|---|
| Base model | `{CHECKPOINT}` |
| Dataset | `{DATASET_NAME}/{DATASET_CFG}` |
| Task | {TASK} |
| Epochs | 3 |
| Batch size | 32 (per device) |
| Learning rate | 2e-5 |
| Weight decay | 0.01 |
| Warmup ratio | 0.1 |
| fp16 | True (GPU) |
| Max sequence length | {MAX_LEN} |
| Best model selection | `load_best_model_at_end=True` on `{METRIC_NAME}` |

## Evaluation

Final `{METRIC_NAME}` on the validation split: **{score_str}**

## Usage

```python
from transformers import pipeline

classifier = pipeline(
    "{TASK}",
    model="{HUB_REPO_ID}",
    truncation=True,
    max_length={MAX_LEN},
)

result = classifier("This movie was fantastic!")
print(result)  # [{{'label': 'POSITIVE', 'score': 0.998}}]
```

## Limitations

- Trained on {DATASET_NAME}/{DATASET_CFG} only — may not generalise to other domains.
- Max input length is {MAX_LEN} tokens; longer inputs are truncated.
- Performance on informal text (tweets, code-switching) has not been evaluated.

## Citation

Generated as part of the **Hugging Face NLP for Engineers** certified-journeys capstone.
"""

card_path = Path(OUTPUT_DIR) / "README.md"
card_path.write_text(model_card)
print(f"\nModel card written to: {card_path}")
print(f"Card length: {len(model_card)} characters")

In [ ]:
# ── Step 6c: Push to Hub ─────────────────────────────────────────────────────
# Uncomment and run this cell after authenticating with notebook_login()

# from huggingface_hub import notebook_login
# notebook_login()   # opens an interactive token input widget

# After logging in, push model + tokenizer + model card:
# trainer.push_to_hub(
#     commit_message=f"Capstone: fine-tuned {CHECKPOINT} on {DATASET_NAME}/{DATASET_CFG}"
# )
# print(f"Model pushed to: https://huggingface.co/{HUB_REPO_ID}")

# Alternative: push the model card separately if trainer.push_to_hub() was already called
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=str(card_path),
#     path_in_repo="README.md",
#     repo_id=HUB_REPO_ID,
#     repo_type="model",
# )

print("Hub push cell is commented out for automated demo execution.")
print("Uncomment the code above after running notebook_login().")
print(f"Your model will appear at: https://huggingface.co/{HUB_REPO_ID}")

### What just happened?
- `trainer.save_model(OUTPUT_DIR)` saves the best checkpoint (selected by `load_best_model_at_end`) — not the final epoch.
- The model card YAML front matter (`---`) is parsed by the Hub to populate the model page metadata — the `model-index` block populates the evaluation leaderboard.
- **`trainer.push_to_hub()` uploads everything in `OUTPUT_DIR`** including `README.md` — no separate upload step needed.
- After pushing, verify the Hub page loads and the usage snippet in the model card runs correctly.

## Stage 7 · Deploy a Gradio Space

A Gradio Space turns your Hub model into a public demo app in under 20 lines of Python. The app:
1. Loads the model from your Hub repo via the `pipeline` API
2. Accepts free-text input from the user
3. Returns predictions with confidence scores for all classes

**To deploy to Spaces:**
1. Create a new Space at [huggingface.co/new-space](https://huggingface.co/new-space)
2. Choose SDK: Gradio, Hardware: CPU Basic (free)
3. Upload `app.py` (the cell below) and `requirements.txt`

The cell below writes `app.py` locally **and** runs the Gradio demo inline in Colab — you can test it before deploying.

In [ ]:
# Write app.py for deployment to a Hugging Face Space
from pathlib import Path

app_code = f'''"""
Gradio demo for {MODEL_SLUG}
Fine-tuned {CHECKPOINT} on {DATASET_NAME}/{DATASET_CFG}.
"""
import gradio as gr
from transformers import pipeline

# Load the model from the Hub — replace with your actual repo ID
HUB_REPO_ID = "{HUB_REPO_ID}"
MAX_LEN     = {MAX_LEN}

# Load once at startup; subsequent calls reuse the cached pipeline
classifier = pipeline(
    "{TASK}",
    model=HUB_REPO_ID,
    truncation=True,
    max_length=MAX_LEN,
    top_k=None,   # return confidence scores for all labels
)

ID2LABEL = classifier.model.config.id2label

def predict(text: str) -> dict:
    """Run inference and return a dict of {{label: score}} for the Gradio Label component."""
    if not text.strip():
        return {{}}
    results = classifier(text)
    # results is a list of {{label, score}} dicts sorted by score descending
    return {{r["label"]: round(r["score"], 4) for r in results}}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type a sentence here…",
        label="Input text",
    ),
    outputs=gr.Label(
        num_top_classes={NUM_LABELS},
        label="Predictions with confidence",
    ),
    title="{MODEL_SLUG}",
    description=(
        "Fine-tuned `{CHECKPOINT}` on `{DATASET_NAME}/{DATASET_CFG}`. "
        f"Enter any text and see the predicted label with confidence scores."
    ),
    examples=[
        ["This movie was absolutely fantastic!"],
        ["I was bitterly disappointed with the service."],
        ["It was neither great nor terrible — just mediocre."],
    ],
    allow_flagging="never",
)

if __name__ == "__main__":
    demo.launch()
'''

Path("app.py").write_text(app_code)

# Write requirements.txt for the Space
Path("requirements.txt").write_text("transformers\ntorch\ngradio\n")

print("Written: app.py")
print("Written: requirements.txt")
print()
print("To deploy to a Space:")
print("  1. Create a new Space at https://huggingface.co/new-space")
print("  2. SDK: Gradio | Hardware: CPU Basic (free)")
print("  3. Upload app.py and requirements.txt")
print(f"  4. Your Space will be live at: https://huggingface.co/spaces/{HF_USERNAME}/{MODEL_SLUG}")

In [ ]:
# ── Run the Gradio demo inline in Colab (local inference) ────────────────────
# This uses the locally-saved model so it works even before Hub push.
import gradio as gr
from transformers import pipeline as hf_pipeline

# Use the locally-saved model; swap OUTPUT_DIR for HUB_REPO_ID after push
local_classifier = hf_pipeline(
    TASK,
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    truncation=True,
    max_length=MAX_LEN,
    top_k=None,
    device="cpu",    # CPU for the demo; GPU is used during training
)

id2label = local_classifier.model.config.id2label

def predict_local(text: str) -> dict:
    if not text.strip():
        return {}
    results = local_classifier(text)
    return {r["label"]: round(r["score"], 4) for r in results}

demo = gr.Interface(
    fn=predict_local,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type a sentence to classify…",
        label="Input text",
    ),
    outputs=gr.Label(
        num_top_classes=NUM_LABELS,
        label="Predictions",
    ),
    title=f"{MODEL_SLUG} — Local Demo",
    description="Running from local checkpoint. Replace OUTPUT_DIR with HUB_REPO_ID after push.",
    examples=[
        ["This movie was absolutely fantastic!"],
        ["I was bitterly disappointed."],
        ["An average experience, nothing special."],
    ],
    allow_flagging="never",
)

# share=True creates a public URL (valid for 72h)
demo.launch(share=False)

### What just happened?
- `gr.Interface` wraps our `predict_local` function with a web UI — input is a `Textbox`, output is a `Label` component that renders confidence bars.
- **`top_k=None` returns all class scores** — the `Label` component displays them as a ranked bar chart.
- `demo.launch(share=False)` starts a local server; set `share=True` to get a public `*.gradio.live` URL.
- The `examples` list pre-populates the demo UI with one-click test inputs — always include at least one positive, one negative, and one ambiguous example.
- After deploying to Spaces, the only change to `app.py` is swapping `OUTPUT_DIR` for `HUB_REPO_ID` in the pipeline call.

## Stage 7b · Verify the Full End-to-End Pipeline

Before declaring the capstone complete, run a programmatic end-to-end check:
1. The local checkpoint loads without errors
2. Inference returns non-empty predictions
3. The model card file exists and contains the key sections
4. (After Hub push) The Hub model is publicly accessible

In [ ]:
from pathlib import Path
import json

def verify_capstone(
    output_dir: str,
    hub_repo_id: str,
    metric_name: str,
    metric_value: float,
) -> None:
    """End-to-end capstone verification checklist."""
    p = Path(output_dir)
    checks = []

    # 1. Artifact files
    checks.append(("config.json saved",        (p / "config.json").exists()))
    checks.append(("tokenizer.json saved",      (p / "tokenizer.json").exists()))
    checks.append(("Model weights saved",
        (p / "pytorch_model.bin").exists() or (p / "model.safetensors").exists()))
    checks.append(("trainer_state.json saved",  (p / "trainer_state.json").exists()))
    checks.append(("README.md (model card)",    (p / "README.md").exists()))
    checks.append(("app.py written",            Path("app.py").exists()))
    checks.append(("requirements.txt written",  Path("requirements.txt").exists()))

    # 2. Metric recorded
    checks.append((f"{metric_name} >= 0.5",    metric_value is not None and metric_value >= 0.5))

    # 3. Model card quality
    if (p / "README.md").exists():
        card_text = (p / "README.md").read_text()
        checks.append(("Model card: training details section", "## Training details" in card_text))
        checks.append(("Model card: evaluation section",       "## Evaluation" in card_text))
        checks.append(("Model card: usage snippet",            "## Usage" in card_text))
        checks.append(("Model card: limitations section",      "## Limitations" in card_text))
    else:
        for section in ["training details", "evaluation", "usage", "limitations"]:
            checks.append((f"Model card: {section}", False))

    # 4. Hub push (manual — cannot verify programmatically without token)
    checks.append((f"Pushed to Hub ({hub_repo_id})", False))  # set True manually after push
    checks.append(("Gradio Space deployed",            False))  # set True manually after deploy
    checks.append(("Inference test on Hub model",      False))  # set True after testing Space

    # Print report
    print("=" * 60)
    print("CAPSTONE VERIFICATION CHECKLIST")
    print("=" * 60)
    passed = 0
    for name, ok in checks:
        icon = "✓" if ok else "○"
        print(f"  {icon}  {name}")
        if ok:
            passed += 1
    print("-" * 60)
    print(f"  {passed}/{len(checks)} checks passed")
    automated = sum(1 for _, ok in checks[:-3] if ok)
    total_auto = len(checks) - 3
    print(f"  {automated}/{total_auto} automated checks | 3 manual checks remain")

verify_capstone(
    output_dir=OUTPUT_DIR,
    hub_repo_id=HUB_REPO_ID,
    metric_name=METRIC_NAME,
    metric_value=primary_score,
)

### What just happened?
- The checklist verifies that every required artifact is present before you declare the capstone complete.
- The **three manual checks** (Hub push, Space deploy, inference test) cannot be automated because they require an active Hub session — mark them `True` manually after completing each step.
- `metric_value >= 0.5` is a floor check, not a target — the real target for SST-2 accuracy is 90%+.
- The model card quality checks confirm the four sections required by the capstone spec are present.

In [ ]:
# Challenge: Full Capstone End-to-End
#
# Complete the capstone by working through all seven stages above with your own
# chosen task. Then wire the remaining pieces together below.
#
# Minimum requirements for a complete capstone submission:
#
# 1. Train on a DIFFERENT dataset or task than the SST-2 default above.
#    Options: imdb (sentiment), conll2003 (NER), ag_news (topic classification),
#    or your own CSV dataset via load_dataset('csv', data_files=...)
#
# 2. Report the CORRECT metric for your task:
#    - Classification (balanced)  → accuracy
#    - Classification (imbalanced) → macro-F1
#    - NER / token classification  → seqeval F1
#    - Summarization               → ROUGE-L
#
# 3. Push your fine-tuned model + tokenizer to the Hub with a complete model card.
#
# 4. Deploy a Gradio Space that loads your Hub model and displays confidence scores.
#
# 5. Complete verify_capstone() with 0 circles (○) remaining.
#
# Scaffold — fill in the blanks:

# ── Your capstone configuration ──────────────────────────────────────────────
MY_USERNAME   = "your-username"          # TODO: your Hub username
MY_CHECKPOINT = "bert-base-uncased"      # TODO: choose distilbert or bert-base
MY_DATASET    = "imdb"                   # TODO: choose a different dataset from default
MY_TASK       = "text-classification"    # TODO: match your dataset
MY_METRIC     = "accuracy"               # TODO: choose the correct metric
MY_SLUG       = "bert-imdb-capstone"     # TODO: descriptive name for your Hub repo

# ── Step 1: Load dataset ─────────────────────────────────────────────────────
# raw_mine = load_dataset(MY_DATASET)
# print(raw_mine)

# ── Step 2: Tokenize ─────────────────────────────────────────────────────────
# tokenizer_mine = AutoTokenizer.from_pretrained(MY_CHECKPOINT)
# tokenized_mine = raw_mine.map(..., batched=True)

# ── Step 3: Train ────────────────────────────────────────────────────────────
# model_mine = AutoModelForSequenceClassification.from_pretrained(
#     MY_CHECKPOINT, num_labels=<your_num_labels>
# )
# training_args_mine = TrainingArguments(
#     output_dir=f"./{MY_SLUG}-output",
#     num_train_epochs=3,
#     fp16=use_fp16,
#     per_device_train_batch_size=16,
#     gradient_accumulation_steps=2,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model=MY_METRIC,
#     report_to="none",
# )
# metric_mine = evaluate.load(MY_METRIC)
# trainer_mine = Trainer(
#     model=model_mine, args=training_args_mine,
#     train_dataset=tokenized_mine["train"],
#     eval_dataset=tokenized_mine["test"],
#     tokenizer=tokenizer_mine,
#     data_collator=DataCollatorWithPadding(tokenizer_mine),
#     compute_metrics=lambda p: metric_mine.compute(
#         predictions=np.argmax(p.predictions, axis=-1), references=p.label_ids
#     ),
# )
# trainer_mine.train()

# ── Step 4: Evaluate ─────────────────────────────────────────────────────────
# results_mine = trainer_mine.evaluate()
# print(results_mine)

# ── Step 5: Save + push ──────────────────────────────────────────────────────
# trainer_mine.save_model(f"./{MY_SLUG}-output")
# tokenizer_mine.save_pretrained(f"./{MY_SLUG}-output")
# # Then: notebook_login() + trainer_mine.push_to_hub()

# ── Step 6: Verify ───────────────────────────────────────────────────────────
# verify_capstone(
#     output_dir=f"./{MY_SLUG}-output",
#     hub_repo_id=f"{MY_USERNAME}/{MY_SLUG}",
#     metric_name=MY_METRIC,
#     metric_value=results_mine[f"eval_{MY_METRIC}"],
# )

print("Capstone scaffold ready. Uncomment the steps above and fill in the TODOs.")

---
## Day 14 key concepts recap

| Concept | What to remember |
|---|---|
| Task selection | Choose dataset + metric pair carefully; wrong metric gives false confidence |
| `DataCollatorWithPadding` | Dynamic padding per batch is more efficient than padding to `MAX_LEN` globally |
| `fp16=True` | Required for GPU training; halves memory and speeds up by 1.5–2× |
| `load_best_model_at_end` | Returns best checkpoint; requires `evaluation_strategy == save_strategy` |
| Test vs. validation | Always evaluate final metrics on a **held-out test split** |
| Model card | YAML front matter + 4 sections: training, evaluation, usage, limitations |
| `push_to_hub()` | Uploads everything in `output_dir` including README.md |
| Gradio `top_k=None` | Returns confidence for all classes — required for the confidence bar chart |
| Gradio Space deployment | Upload `app.py` + `requirements.txt` to a new Space; CPU Basic is free |
| Verify the Hub copy | Always run an inference test on the **deployed** model, not the in-memory one |

> **Tip:** For the capstone, document every training decision in your model card: why you chose the base model, what learning rate and batch size you used, and what the final eval metric was — this is what makes a Hub model genuinely useful to others.

---
## Course complete — what you built

Over 14 days you went from Hugging Face basics to a deployed NLP system:

- **Days 1–6:** Transformers fundamentals, tokenizers, datasets, pipelines
- **Days 7–10:** Fine-tuning with Trainer, custom metrics, LoRA/PEFT
- **Days 11–12:** Evaluation strategies, Hub integration, Gradio
- **Day 13:** Production readiness — pitfalls, OOM debugging, checklists
- **Day 14:** Full capstone — fine-tune → evaluate → Hub → Gradio Space

Mark Day 14 complete in your [tracker](../index.html) and claim your course badge!